# Annotation manuelle — validation réelle

1. Exécute la cellule suivante.
2. Pour une image **visible**, trace un rectangle à la souris autour du support cible puis clique **Visible + enregistrer**.
3. Si le support n'est pas réellement identifiable : **Non visible**.
4. Si tu ne peux pas décider proprement : **Ambigu**.

`ambigu` sera exclu des métriques. `non_visible` est conservé comme image négative.

In [ ]:
%matplotlib widget
from pathlib import Path
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.widgets import RectangleSelector
import ipywidgets as widgets
from IPython.display import display, clear_output

candidates = [Path.cwd(), Path.cwd() / 'baseline_yolo']
ROOT = next((p for p in candidates if (p/'data/niveau2_manual_holdout_final/annotations.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Impossible de trouver data/niveau2_manual_holdout_final/annotations.csv')

CSV = ROOT / 'data/niveau2_manual_holdout_final/annotations.csv'
IMG_DIR = ROOT / 'data/niveau2_manual_holdout_final/images'
df = pd.read_csv(CSV, dtype={'SUP_ID':'string'})
df["status"] = df["status"].astype("object")

if "commentaire" in df.columns:
    df["commentaire"] = df["commentaire"].astype("object")
# Les colonnes texte sont entièrement vides au départ.
# Pandas peut sinon les interpréter comme float64.
df["status"] = df["status"].fillna("").astype("object")
df["commentaire"] = df["commentaire"].fillna("").astype("object")
print('Annotations à réaliser :', len(df))
print('Déjà renseignées :', df['status'].fillna('').astype(str).str.strip().ne('').sum())


In [ ]:
current = 0
selected_bbox = None
selector = None

fig, ax = plt.subplots(figsize=(8, 8))

out = widgets.Output()
btn_prev = widgets.Button(description='← Précédente')
btn_visible = widgets.Button(description='Visible + enregistrer', button_style='success')
btn_nonvisible = widgets.Button(description='Non visible', button_style='warning')
btn_ambigu = widgets.Button(description='Ambigu')
btn_next = widgets.Button(description='Suivante →')

def _as_float(v):
    try:
        if pd.isna(v):
            return None
        return float(v)
    except Exception:
        return None

def onselect(eclick, erelease):
    global selected_bbox
    x1, x2 = sorted([float(eclick.xdata), float(erelease.xdata)])
    y1, y2 = sorted([float(eclick.ydata), float(erelease.ydata)])
    selected_bbox = (x1, y1, x2, y2)
    with out:
        clear_output(wait=True)
        print('BBox sélectionnée :', tuple(round(v, 1) for v in selected_bbox))

def show_image():
    global selector, selected_bbox
    row = df.iloc[current]
    path = IMG_DIR / row['image']
    im = Image.open(path).convert('RGB')
    ax.clear()
    ax.imshow(im)
    # Zone de recherche autour de la coordonnée ANFR
    cx, cy = 512, 512
    taille = 160

    guide = Rectangle(
    (cx - taille / 2, cy - taille / 2),
    taille,
    taille,
    fill=False,
    edgecolor="red",
    linewidth=2,
    linestyle="--"
    )

    ax.add_patch(guide)
    status = '' if pd.isna(row.get('status', '')) else str(row.get('status', '')).strip()
    ax.set_title(f"{current+1}/{len(df)} | SUP_ID {row['SUP_ID']} | {row['NAT_LB_NOM']} | status={status or 'NON ANNOTE'}")
    ax.axis('off')

    vals = [_as_float(row.get(c)) for c in ['x1','y1','x2','y2']]
    if status == 'visible' and all(v is not None for v in vals):
        x1, y1, x2, y2 = vals
        selected_bbox = (x1, y1, x2, y2)
        ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False, edgecolor='lime', linewidth=2))
    else:
        selected_bbox = None

    selector = RectangleSelector(
        ax,
        onselect,
        useblit=True,
        button=[1],
        minspanx=5,
        minspany=5,
        spancoords='pixels',
        interactive=True,
    )
    fig.canvas.draw_idle()
    with out:
        clear_output(wait=True)
        print('Trace une bbox si visible, puis utilise les boutons.')

def save_row(status):
    global current
    if status == 'visible':
        if selected_bbox is None:
            with out:
                clear_output(wait=True)
                print('⚠️ Trace d’abord un rectangle.')
            return
        x1, y1, x2, y2 = selected_bbox
        df.loc[current, ['status','x1','y1','x2','y2']] = [status, x1, y1, x2, y2]
    else:
        df.loc[current, 'status'] = status
        df.loc[current, ['x1','y1','x2','y2']] = [pd.NA, pd.NA, pd.NA, pd.NA]
    df.to_csv(CSV, index=False)
    if current < len(df)-1:
        current += 1
    show_image()

def prev(_):
    global current
    current = max(0, current-1)
    show_image()

def next_(_):
    global current
    current = min(len(df)-1, current+1)
    show_image()

btn_prev.on_click(prev)
btn_next.on_click(next_)
btn_visible.on_click(lambda _: save_row('visible'))
btn_nonvisible.on_click(lambda _: save_row('non_visible'))
btn_ambigu.on_click(lambda _: save_row('ambigu'))

display(widgets.HBox([btn_prev, btn_visible, btn_nonvisible, btn_ambigu, btn_next]))
display(out)
show_image()


In [ ]:
# Contrôle final
df = pd.read_csv(CSV, dtype={'SUP_ID':'string'})
print(df['status'].fillna('NON_ANNOTE').value_counts())
non_annotes = df[df['status'].fillna('').astype(str).str.strip() == '']
print('Non annotés :', len(non_annotes))
if len(non_annotes) == 0:
    print('✅ Annotations entraînement Niveau 2 terminée.')


In [ ]:
import pandas as pd

print("===== CORRECTION DES DATAFRAMES =====")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame) and "status" in obj.columns:

        print(
            name,
            "| avant :",
            obj["status"].dtype
        )

        obj["status"] = (
            obj["status"]
            .fillna("")
            .astype("object")
        )

        if "commentaire" in obj.columns:
            obj["commentaire"] = (
                obj["commentaire"]
                .fillna("")
                .astype("object")
            )

        print(
            name,
            "| après :",
            obj["status"].dtype
        )